In [ ]:
import pandas as pd

df = pd.read_csv("../data/processed/cleaned_reviews.csv")

print(df.shape)
df.head()

In [1]:
import pandas as pd

test_df = pd.read_csv(
    "../data/processed/cleaned_reviews.csv",
    nrows=5
)

print(test_df.shape)
test_df

(5, 9)


,Product_name,Price,Rate,Review,Summary,full_review,sentiment,review_length,word_count
0,"Crompton 75 L Desert Air Cooler(White, Teal, A...","??10,499",5.0,Simply awesome,it's really worth every single penny. it works...,Simply awesome it's really worth every single ...,Positive,508,92
1,"Crompton 75 L Desert Air Cooler(White, Teal, A...","??10,499",4.0,Worth the money . Desert Cooler live up to the...,I bought Crompton Ozone 75 Desert Air Cooler i...,Worth the money . Desert Cooler live up to the...,Positive,548,112
2,"Crompton 75 L Desert Air Cooler(White, Teal, A...","??10,499",5.0,Worth every penny,GREAT packaging by seller. As this was the mos...,Worth every penny GREAT packaging by seller. A...,Positive,518,89
3,"Crompton 75 L Desert Air Cooler(White, Teal, A...","??10,499",5.0,Fabulous!,Delivery was delayed by two days except this e...,Fabulous! Delivery was delayed by two days exc...,Positive,414,69
4,"Crompton 75 L Desert Air Cooler(White, Teal, A...","??10,499",4.0,Nice product,A Good cooler by Crompton. The height of the c...,Nice product A Good cooler by Crompton. The he...,Positive,515,102


In [2]:
df = pd.read_csv(
    "../data/processed/cleaned_reviews.csv",
    encoding="utf-8"
)

print("Loaded successfully")
print(df.shape)

Loaded successfully
(306316, 9)


In [3]:
import re
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [4]:
lemmatizer = WordNetLemmatizer()

stop_words = set(stopwords.words("english"))

stop_words = stop_words - {"no", "not", "nor"}

In [5]:
def preprocess_text(text):
    text = text.lower()

    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    tokens = word_tokenize(text)

    tokens = [
        word for word in tokens
        if word not in stop_words
    ]

    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    return " ".join(tokens)

In [6]:
print(df["full_review"].iloc[0])

print("\nAfter preprocessing:")

print(preprocess_text(df["full_review"].iloc[0]))

Simply awesome it's really worth every single penny. it works like one ton AC provided that your room has proper ventilation.  I generally keep cooler near to window so that it can pull fresh air from outside and as a result I get very cool air. one more suggestion,  please don't buy cooler which are equipped with a blower fan. Always go for the cooler equipped with metal fan like this one.  In one word, this Compton cooler is Amazing. if you find the review helpful, please thumps up. Happy summer! ????

After preprocessing:
simply awesome really worth every single penny work like one ton ac provided room proper ventilation generally keep cooler near window pull fresh air outside result get cool air one suggestion please buy cooler equipped blower fan always go cooler equipped metal fan like one one word compton cooler amazing find review helpful please thump happy summer


In [7]:
df["clean_text"] = df["full_review"].apply(preprocess_text)

In [8]:
print(df.shape)
print(df.columns.tolist())

(306316, 10)
['Product_name', 'Price', 'Rate', 'Review', 'Summary', 'full_review', 'sentiment', 'review_length', 'word_count', 'clean_text']


In [9]:
df.to_csv(
    "../data/processed/nlp_processed_reviews.csv",
    index=False
)

In [10]:
import os

print(
    os.path.exists(
        "../data/processed/nlp_processed_reviews.csv"
    )
)

True


In [11]:
y = df["sentiment"]

print(y.head())

0    Positive
1    Positive
2    Positive
3    Positive
4    Positive
Name: sentiment, dtype: str


In [12]:
print(y.shape)

(306316,)


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [14]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.90
)

In [15]:
X = tfidf.fit_transform(df["clean_text"])

In [16]:
print(X.shape)

(306316, 5000)


In [17]:
y = df["sentiment"]

X = tfidf.fit_transform(df["clean_text"])

print("X:", X.shape)
print("y:", y.shape)

X: (306316, 5000)
y: (306316,)


In [18]:
from sklearn.model_selection import train_test_split

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [20]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (245052, 5000)
X_test : (61264, 5000)
y_train: (245052,)
y_test : (61264,)


In [21]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000
)

In [22]:
model.fit(X_train, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [23]:
print(model.classes_)

['Negative' 'Neutral' 'Positive']


In [24]:
prediction = model.predict(X_test[0])

print(prediction)

['Positive']


In [25]:
print("Predicted:", model.predict(X_test[0])[0])
print("Actual   :", y_test.iloc[0])

Predicted: Positive
Actual   : Positive


In [26]:
index = y_test.index[0]

print("REVIEW:")
print(df.loc[index, "full_review"])

print("\nACTUAL:")
print(y_test.iloc[0])

print("\nPREDICTED:")
print(model.predict(X_test[0])[0])

REVIEW:
Just wow! It's amazing

ACTUAL:
Positive

PREDICTED:
Positive


In [27]:
probabilities = model.predict_proba(X_test[0])[0]

for sentiment, probability in zip(model.classes_, probabilities):
    print(sentiment, ":", round(probability * 100, 2), "%")

Negative : 0.05 %
Neutral : 0.16 %
Positive : 99.79 %


In [28]:
y_pred = model.predict(X_test)

In [29]:
print(y_pred.shape)

(61264,)


In [30]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.9819796291459911


In [31]:
print("Accuracy:", round(accuracy * 100, 2), "%")

Accuracy: 98.2 %


In [32]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

    Negative       0.99      0.99      0.99      9924
     Neutral       0.95      0.87      0.91      5570
    Positive       0.98      0.99      0.99     45770

    accuracy                           0.98     61264
   macro avg       0.97      0.95      0.96     61264
weighted avg       0.98      0.98      0.98     61264



In [33]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    Negative       0.99      0.99      0.99      9924
     Neutral       0.95      0.87      0.91      5570
    Positive       0.98      0.99      0.99     45770

    accuracy                           0.98     61264
   macro avg       0.97      0.95      0.96     61264
weighted avg       0.98      0.98      0.98     61264



In [34]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=model.classes_
)

print(model.classes_)
print(cm)

['Negative' 'Neutral' 'Positive']
[[ 9801    37    86]
 [   68  4870   632]
 [   51   230 45489]]


In [35]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=model.classes_
)

print(model.classes_)
print(cm)

['Negative' 'Neutral' 'Positive']
[[ 9801    37    86]
 [   68  4870   632]
 [   51   230 45489]]


In [36]:
X_text = df["clean_text"]
y = df["sentiment"]

In [37]:
from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [38]:
print("Training text:", X_train_text.shape)
print("Testing text :", X_test_text.shape)

Training text: (245052,)
Testing text : (61264,)


In [39]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_final = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.90
)

In [40]:
X_train = tfidf_final.fit_transform(X_train_text)

In [41]:
X_test = tfidf_final.transform(X_test_text)

In [42]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

X_train: (245052, 5000)
X_test : (61264, 5000)


In [43]:
from sklearn.linear_model import LogisticRegression

final_model = LogisticRegression(
    max_iter=1000
)

final_model.fit(X_train, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [44]:
y_pred = final_model.predict(X_test)

In [45]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print(
    "Final Accuracy:",
    round(accuracy * 100, 2),
    "%"
)

Final Accuracy: 98.19 %


In [46]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

    Negative       0.99      0.99      0.99      9924
     Neutral       0.95      0.88      0.91      5570
    Positive       0.98      0.99      0.99     45770

    accuracy                           0.98     61264
   macro avg       0.97      0.95      0.96     61264
weighted avg       0.98      0.98      0.98     61264



In [47]:
def predict_sentiment(review):

    cleaned = preprocess_text(review)

    vector = tfidf_final.transform([cleaned])

    prediction = final_model.predict(vector)[0]

    probabilities = final_model.predict_proba(vector)[0]

    return prediction, probabilities

In [48]:
review = "This product is absolutely terrible. Waste of money and stopped working."

prediction, probabilities = predict_sentiment(review)

print("Prediction:", prediction)

for sentiment, probability in zip(final_model.classes_, probabilities):
    print(
        sentiment,
        ":",
        round(probability * 100, 2),
        "%"
    )

Prediction: Negative
Negative : 99.88 %
Neutral : 0.06 %
Positive : 0.06 %


In [49]:
review = "Amazing product. Excellent quality and totally worth the money."

prediction, probabilities = predict_sentiment(review)

print("Prediction:", prediction)

for sentiment, probability in zip(final_model.classes_, probabilities):
    print(sentiment, ":", round(probability * 100, 2), "%")

Prediction: Positive
Negative : 0.05 %
Neutral : 0.09 %
Positive : 99.86 %


In [50]:
review = "The product is okay. Nothing special but it works."

prediction, probabilities = predict_sentiment(review)

print("Prediction:", prediction)

for sentiment, probability in zip(final_model.classes_, probabilities):
    print(sentiment, ":", round(probability * 100, 2), "%")

Prediction: Neutral
Negative : 6.35 %
Neutral : 61.7 %
Positive : 31.96 %


In [51]:
import os

os.makedirs("../models", exist_ok=True)

In [52]:
import joblib

In [53]:
joblib.dump(
    final_model,
    "../models/sentiment_model.pkl"
)

['../models/sentiment_model.pkl']

In [54]:
joblib.dump(
    tfidf_final,
    "../models/tfidf_vectorizer.pkl"
)

['../models/tfidf_vectorizer.pkl']

In [55]:
print(
    os.path.exists(
        "../models/sentiment_model.pkl"
    )
)

print(
    os.path.exists(
        "../models/tfidf_vectorizer.pkl"
    )
)

True
True


In [56]:
loaded_model = joblib.load(
    "../models/sentiment_model.pkl"
)

loaded_tfidf = joblib.load(
    "../models/tfidf_vectorizer.pkl"
)

In [57]:
print(loaded_model.classes_)

['Negative' 'Neutral' 'Positive']


In [1]:
import numpy as np

A = np.array([2, 2])
B = np.array([4, 4])

dot_product = np.dot(A, B)

magnitude_A = np.linalg.norm(A)
magnitude_B = np.linalg.norm(B)

cosine_similarity = dot_product / (
    magnitude_A * magnitude_B
)

print(cosine_similarity)

0.9999999999999998


In [2]:
A = np.array([2, 2])
C = np.array([-2, -2])

cosine_similarity = np.dot(A, C) / (
    np.linalg.norm(A) * np.linalg.norm(C)
)

print(cosine_similarity)

-0.9999999999999998


In [3]:
from sentence_transformers import SentenceTransformer

c:\Users\RUSHIKESH\Desktop\customer-review-intelligence\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\RUSHIKESH\Desktop\customer-review-intelligence\venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\RUSHIKESH\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6789.25it

In [5]:
sentence = "Battery drains very quickly."

In [6]:
embedding = embedding_model.encode(sentence)

In [7]:
embedding

array([ 4.23443802e-02,  6.84453771e-02,  4.07577083e-02,  2.05847397e-02,
        2.24376526e-02, -4.76915054e-02, -2.66008545e-02,  2.18102392e-02,
        1.00337379e-01,  5.70784546e-02, -1.96329765e-02,  6.01917505e-02,
       -3.63771170e-02,  1.01048991e-01,  1.37857655e-02,  1.65194701e-02,
        1.61430258e-02, -2.51111817e-02, -2.82504074e-02, -5.43586984e-02,
        6.43687323e-02, -7.22889975e-02,  3.80654968e-02,  3.37671442e-03,
        6.25481829e-02,  3.40962633e-02,  1.15452548e-02, -2.06394680e-02,
       -3.13156843e-02, -5.83341792e-02, -8.76207128e-02,  6.41768128e-02,
        7.47050298e-03, -1.80540625e-02,  4.22609448e-02,  4.73440513e-02,
       -7.92872831e-02,  9.30086747e-02,  4.73331921e-02, -6.83159754e-02,
       -7.01188575e-03,  3.78395617e-02,  1.04748763e-01, -2.38667578e-02,
        3.87798622e-02,  3.90970223e-02, -6.17843401e-03, -1.50071252e-02,
        9.02467072e-02, -4.14711051e-03,  7.53359050e-02,  1.21283280e-02,
       -1.16332783e-03, -

In [8]:
print(embedding.shape)

(384,)


In [9]:
print(embedding[:10])

[ 0.04234438  0.06844538  0.04075771  0.02058474  0.02243765 -0.04769151
 -0.02660085  0.02181024  0.10033738  0.05707845]


In [10]:
sentences = [
    "Battery drains very quickly.",
    "The phone does not hold charge for long.",
    "The camera takes beautiful photos."
]

embeddings = embedding_model.encode(sentences)

In [11]:
print(embeddings.shape)

(3, 384)


In [12]:
A = embeddings[0]
B = embeddings[1]
C = embeddings[2]

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
similarity_AB = cosine_similarity([A], [B])[0][0]

print(similarity_AB)

0.35002488


In [15]:
similarity_AC = cosine_similarity([A], [C])[0][0]

print(similarity_AC)

0.11947813


In [16]:
print(
    "Battery vs Charge:",
    round(similarity_AB, 4)
)

print(
    "Battery vs Camera:",
    round(similarity_AC, 4)
)

Battery vs Charge: 0.35
Battery vs Camera: 0.1195


In [17]:
import chromadb

In [18]:
client = chromadb.Client()

In [19]:
collection = client.create_collection(
    name="demo_reviews"
)

In [20]:
reviews = [
    "Battery drains very quickly.",
    "The phone needs charging several times a day.",
    "Camera quality is excellent.",
    "The delivery arrived very late.",
    "Battery backup became poor after the update.",
    "The display is bright and beautiful."
]

In [25]:
ids = [
    "review_1",
    "review_2",
    "review_3",
    "review_4",
    "review_5",
    "review_6"
]

print(ids)

['review_1', 'review_2', 'review_3', 'review_4', 'review_5', 'review_6']


In [21]:
review_embeddings = embedding_model.encode(
    reviews
)

In [26]:
print(len(reviews))
print(len(review_embeddings_list))
print(len(metadatas))
print(len(ids))

6


NameError: name 'review_embeddings_list' is not defined

In [22]:
print(review_embeddings.shape)

(6, 384)


In [23]:
metadatas = [
    {"topic": "battery"},
    {"topic": "battery"},
    {"topic": "camera"},
    {"topic": "delivery"},
    {"topic": "battery"},
    {"topic": "display"}
]

In [24]:
collection.add(
    ids=ids,
    documents=reviews,
    embeddings=review_embeddings_list,
    metadatas=metadatas
)

NameError: name 'ids' is not defined

In [27]:
print(embedding_model)

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


In [28]:
review_embeddings_list = review_embeddings.tolist()

In [29]:
print(len(review_embeddings_list))
print(len(review_embeddings_list[0]))

6
384


In [30]:
ids = [
    "review_1",
    "review_2",
    "review_3",
    "review_4",
    "review_5",
    "review_6"
]

metadatas = [
    {"topic": "battery"},
    {"topic": "battery"},
    {"topic": "camera"},
    {"topic": "delivery"},
    {"topic": "battery"},
    {"topic": "display"}
]

In [31]:
print("Reviews:", len(reviews))
print("Embeddings:", len(review_embeddings_list))
print("Metadata:", len(metadatas))
print("IDs:", len(ids))

Reviews: 6
Embeddings: 6
Metadata: 6
IDs: 6


In [32]:
collection.add(
    ids=ids,
    documents=reviews,
    embeddings=review_embeddings_list,
    metadatas=metadatas
)

print("Records stored:", collection.count())

Records stored: 6


In [33]:
query = "What problems are users having with charging?"

In [34]:
query_embedding = embedding_model.encode(query)

print(query_embedding.shape)

(384,)


In [35]:
results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3
)

In [36]:
print(results["documents"][0])

['The phone needs charging several times a day.', 'Battery drains very quickly.', 'Battery backup became poor after the update.']


In [37]:
documents = results["documents"][0]
distances = results["distances"][0]

for i, (document, distance) in enumerate(
    zip(documents, distances),
    start=1
):
    print(f"Result {i}")
    print("Review:", document)
    print("Distance:", round(distance, 4))
    print()

Result 1
Review: The phone needs charging several times a day.
Distance: 1.0629

Result 2
Review: Battery drains very quickly.
Distance: 1.2613

Result 3
Review: Battery backup became poor after the update.
Distance: 1.4576



In [38]:
query = "How is the picture and camera quality?"

query_embedding = embedding_model.encode(query)

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3
)

documents = results["documents"][0]
distances = results["distances"][0]

for i, (document, distance) in enumerate(
    zip(documents, distances),
    start=1
):
    print(f"Result {i}")
    print("Review:", document)
    print("Distance:", round(distance, 4))
    print()

Result 1
Review: Camera quality is excellent.
Distance: 0.407

Result 2
Review: The display is bright and beautiful.
Distance: 1.3061

Result 3
Review: Battery drains very quickly.
Distance: 1.7309



In [39]:
print(df.shape)

NameError: name 'df' is not defined

In [40]:
import pandas as pd

df = pd.read_csv("../data/processed/nlp_processed_reviews.csv")

In [41]:
print(df.shape)
print(df.columns)

(306316, 10)
Index(['Product_name', 'Price', 'Rate', 'Review', 'Summary', 'full_review',
       'sentiment', 'review_length', 'word_count', 'clean_text'],
      dtype='str')


In [42]:
df[["Product_name", "Rate", "full_review", "sentiment"]].isnull().sum()

Product_name    0
Rate            0
full_review     0
sentiment       0
dtype: int64

In [43]:
rag_df = df[
    ["Product_name", "Rate", "full_review", "sentiment"]
].sample(
    n=1000,
    random_state=42
).reset_index(drop=True)

print(rag_df.shape)

(1000, 4)


In [44]:
rag_df.head(3)

,Product_name,Rate,full_review,sentiment
0,PHILIPS MMS8085B/94 Convertible 80 W Bluetooth...,4.0,Good quality product a good quality product fr...,Positive
1,Crompton 88 L Desert Air Cooler with Honeycomb...,4.0,Good quality product Ok,Positive
2,Cosito 144 TC Cotton Double Floral Flat Bedshe...,5.0,Mind-blowing purchase The product is very nice...,Positive


In [45]:
documents = rag_df["full_review"].tolist()

print("Number of documents:", len(documents))
print()
print("First document:")
print(documents[0])

Number of documents: 1000

First document:
Good quality product a good quality product from philips.unexpectable bass..awsome..


In [46]:
real_embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True
)

print(real_embeddings.shape)

Batches: 100%|██████████| 32/32 [00:05<00:00,  5.37it/s]

(1000, 384)


In [47]:
real_ids = [
    f"real_review_{i}"
    for i in range(len(rag_df))
]

print("Number of IDs:", len(real_ids))
print("First 5 IDs:", real_ids[:5])

Number of IDs: 1000
First 5 IDs: ['real_review_0', 'real_review_1', 'real_review_2', 'real_review_3', 'real_review_4']


In [48]:
first_metadata = {
    "product": str(rag_df.loc[0, "Product_name"]),
    "rating": float(rag_df.loc[0, "Rate"]),
    "sentiment": str(rag_df.loc[0, "sentiment"])
}

print(first_metadata)

{'product': 'PHILIPS MMS8085B/94 Convertible 80 W Bluetooth Home Theatre(Black, 2.1 Channel)', 'rating': 4.0, 'sentiment': 'Positive'}


In [49]:
real_metadatas = []

for _, row in rag_df.iterrows():
    real_metadatas.append({
        "product": str(row["Product_name"]),
        "rating": float(row["Rate"]),
        "sentiment": str(row["sentiment"])
    })

print("Number of metadata records:", len(real_metadatas))
print()
print("First metadata:")
print(real_metadatas[0])

Number of metadata records: 1000

First metadata:
{'product': 'PHILIPS MMS8085B/94 Convertible 80 W Bluetooth Home Theatre(Black, 2.1 Channel)', 'rating': 4.0, 'sentiment': 'Positive'}


In [50]:
print("Documents :", len(documents))
print("Embeddings:", len(real_embeddings))
print("IDs       :", len(real_ids))
print("Metadata  :", len(real_metadatas))

Documents : 1000
Embeddings: 1000
IDs       : 1000
Metadata  : 1000


In [51]:
import chromadb

persistent_client = chromadb.PersistentClient(
    path="../vector_db"
)

In [52]:
print(persistent_client)

In [53]:
real_collection = persistent_client.get_or_create_collection(
    name="customer_reviews"
)

print("Records currently stored:", real_collection.count())

Records currently stored: 0


In [54]:
real_collection.add(
    ids=real_ids,
    documents=documents,
    embeddings=real_embeddings.tolist(),
    metadatas=real_metadatas
)

In [55]:
print("Records stored:", real_collection.count())

Records stored: 1000


In [56]:
query = "Customers complaining about poor product quality"

query_embedding = embedding_model.encode(query)

print(query_embedding.shape)


(384,)


In [57]:
results = real_collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5
)

In [58]:
for i, review in enumerate(
    results["documents"][0],
    start=1
):
    print(f"Result {i}:")
    print(review)
    print()

Result 1:
Bad quality Bad quality product

Result 2:
Expected a better product Not good

Result 3:
Good quality product Low money good product

Result 4:
Unsatisfactory - Poor Product  + Poor Customer Service I have to purchase last month 16th Oct  and with in 1 month its showing error in system in this radio, i have to complaint but already 7 days gone, no one can give the exactly response.

Result 5:
Good quality product Quality is good.



In [59]:
for i, (review, metadata) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0]
    ),
    start=1
):
    print(f"Result {i}")
    print("Review:", review)
    print("Sentiment:", metadata["sentiment"])
    print()

Result 1
Review: Bad quality Bad quality product
Sentiment: Negative

Result 2
Review: Expected a better product Not good
Sentiment: Negative

Result 3
Review: Good quality product Low money good product
Sentiment: Positive

Result 4
Review: Unsatisfactory - Poor Product  + Poor Customer Service I have to purchase last month 16th Oct  and with in 1 month its showing error in system in this radio, i have to complaint but already 7 days gone, no one can give the exactly response.
Sentiment: Negative

Result 5
Review: Good quality product Quality is good.
Sentiment: Positive



In [60]:
negative_results = real_collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5,
    where={"sentiment": "Negative"}
)

In [61]:
for i, review in enumerate(
    negative_results["documents"][0],
    start=1
):
    print(f"Result {i}:")
    print(review)
    print()

Result 1:
Bad quality Bad quality product

Result 2:
Expected a better product Not good

Result 3:
Unsatisfactory - Poor Product  + Poor Customer Service I have to purchase last month 16th Oct  and with in 1 month its showing error in system in this radio, i have to complaint but already 7 days gone, no one can give the exactly response.

Result 4:
Horrible waste of money , quality is very bad

Result 5:
Did not meet expectations Worst qualityPlease don't buy



In [62]:
for i, (review, distance) in enumerate(
    zip(
        negative_results["documents"][0],
        negative_results["distances"][0]
    ),
    start=1
):
    print(f"Result {i}")
    print("Review:", review)
    print("Distance:", round(distance, 4))
    print()

Result 1
Review: Bad quality Bad quality product
Distance: 0.5401

Result 2
Review: Expected a better product Not good
Distance: 0.8026

Result 3
Review: Unsatisfactory - Poor Product  + Poor Customer Service I have to purchase last month 16th Oct  and with in 1 month its showing error in system in this radio, i have to complaint but already 7 days gone, no one can give the exactly response.
Distance: 0.813

Result 4
Review: Horrible waste of money , quality is very bad
Distance: 0.8148

Result 5
Review: Did not meet expectations Worst qualityPlease don't buy
Distance: 0.8335



In [63]:
print(
    "Unique products:",
    rag_df["Product_name"].nunique()
)

Unique products: 392


In [64]:
full_collection = persistent_client.get_or_create_collection(
    name="all_customer_reviews"
)

print("Currently stored:", full_collection.count())

Currently stored: 0


In [65]:
batch_size = 1000

print("Batch size:", batch_size)
print("Total reviews:", len(df))

Batch size: 1000
Total reviews: 306316


In [66]:
start = 0
end = start + batch_size

batch_df = df.iloc[start:end]

print("Batch shape:", batch_df.shape)
print("Start:", start)
print("End:", end)

Batch shape: (1000, 10)
Start: 0
End: 1000


In [67]:
batch_documents = batch_df["full_review"].astype(str).tolist()

print("Documents:", len(batch_documents))
print()
print("First document:")
print(batch_documents[0])

Documents: 1000

First document:
Simply awesome it's really worth every single penny. it works like one ton AC provided that your room has proper ventilation.  I generally keep cooler near to window so that it can pull fresh air from outside and as a result I get very cool air. one more suggestion,  please don't buy cooler which are equipped with a blower fan. Always go for the cooler equipped with metal fan like this one.  In one word, this Compton cooler is Amazing. if you find the review helpful, please thumps up. Happy summer! ????


batch_embeddings = embedding_model.encode(
    batch_documents,
    show_progress_bar=True
)

print("Embedding shape:", batch_embeddings.shape)

In [68]:
batch_embeddings = embedding_model.encode(
    batch_documents,
    show_progress_bar=True
)

print("Embedding shape:", batch_embeddings.shape)

Batches: 100%|██████████| 32/32 [00:06<00:00,  4.66it/s]

Embedding shape: (1000, 384)


In [69]:

batch_ids = [
    f"review_{i}"
    for i in range(start, end)
]

print("Number of IDs:", len(batch_ids))
print("First 3:", batch_ids[:3])
print("Last 3:", batch_ids[-3:])

Number of IDs: 1000
First 3: ['review_0', 'review_1', 'review_2']
Last 3: ['review_997', 'review_998', 'review_999']


In [70]:
batch_metadatas = []

for _, row in batch_df.iterrows():
    batch_metadatas.append({
        "product": str(row["Product_name"]),
        "rating": float(row["Rate"]),
        "sentiment": str(row["sentiment"])
    })

print("Metadata records:", len(batch_metadatas))
print()
print("First metadata:")
print(batch_metadatas[0])

Metadata records: 1000

First metadata:
{'product': 'Crompton 75 L Desert Air Cooler(White, Teal, ACGC-DAC751)', 'rating': 5.0, 'sentiment': 'Positive'}


In [71]:
full_collection.add(
    ids=batch_ids,
    documents=batch_documents,
    embeddings=batch_embeddings.tolist(),
    metadatas=batch_metadatas
)

In [72]:
print("Total records stored:", full_collection.count())

Total records stored: 1000


In [73]:
total_rows = len(df)
batch_size = 1000

for start in range(1000, total_rows, batch_size):
    end = min(start + batch_size, total_rows)

    batch_df = df.iloc[start:end]

    batch_documents = (
        batch_df["full_review"]
        .astype(str)
        .tolist()
    )

    batch_embeddings = embedding_model.encode(
        batch_documents,
        show_progress_bar=False
    )

    batch_ids = [
        f"review_{i}"
        for i in range(start, end)
    ]

    batch_metadatas = []

    for _, row in batch_df.iterrows():
        batch_metadatas.append({
            "product": str(row["Product_name"]),
            "rating": float(row["Rate"]),
            "sentiment": str(row["sentiment"])
        })

    full_collection.add(
        ids=batch_ids,
        documents=batch_documents,
        embeddings=batch_embeddings.tolist(),
        metadatas=batch_metadatas
    )

    print(
        f"Stored {end}/{total_rows} reviews"
    )

Stored 2000/306316 reviews
Stored 3000/306316 reviews
Stored 4000/306316 reviews
Stored 5000/306316 reviews
Stored 6000/306316 reviews
Stored 7000/306316 reviews
Stored 8000/306316 reviews
Stored 9000/306316 reviews
Stored 10000/306316 reviews
Stored 11000/306316 reviews
Stored 12000/306316 reviews
Stored 13000/306316 reviews
Stored 14000/306316 reviews
Stored 15000/306316 reviews
Stored 16000/306316 reviews
Stored 17000/306316 reviews
Stored 18000/306316 reviews
Stored 19000/306316 reviews
Stored 20000/306316 reviews
Stored 21000/306316 reviews
Stored 22000/306316 reviews
Stored 23000/306316 reviews
Stored 24000/306316 reviews
Stored 25000/306316 reviews
Stored 26000/306316 reviews
Stored 27000/306316 reviews
Stored 28000/306316 reviews
Stored 29000/306316 reviews
Stored 30000/306316 reviews
Stored 31000/306316 reviews
Stored 32000/306316 reviews
Stored 33000/306316 reviews
Stored 34000/306316 reviews
Stored 35000/306316 reviews
Stored 36000/306316 reviews
Stored 37000/306316 reviews


In [74]:
print("Final records:", full_collection.count())

Final records: 306316


In [75]:
query = "Air cooler is not cooling the room properly"

query_embedding = embedding_model.encode(query)

print(query_embedding.shape)

(384,)


In [76]:
full_results = full_collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=10
)

In [77]:
for i, (review, metadata, distance) in enumerate(
    zip(
        full_results["documents"][0],
        full_results["metadatas"][0],
        full_results["distances"][0]
    ),
    start=1
):
    print(f"Result {i}")
    print("Review:", review)
    print("Product:", metadata["product"])
    print("Sentiment:", metadata["sentiment"])
    print("Rating:", metadata["rating"])
    print("Distance:", round(distance, 4))
    print("-" * 50)

Result 1
Review: Moderate This air cooler does not proper cooling than other metal body cooler,
Product: Crompton 75 L Desert Air Cooler(White, Teal, ACGC-DAC751)
Sentiment: Negative
Rating: 2.0
Distance: 0.5032
--------------------------------------------------
Result 2
Review: Moderate This air cooler does not proper cooling than other metal body cooler,
Product: Crompton 88 L Desert Air Cooler with Honeycomb Cooling Pad(White, Teal, ACGC-DAC881)
Sentiment: Negative
Rating: 2.0
Distance: 0.5032
--------------------------------------------------
Result 3
Review: Nice Entire room is not cooling even kept for day and night.avg cooling.review after use of 15 days.normal fan type.
Product: Symphony 27 L Room/Personal Air Cooler(White, Blue, Ice Cube 27)
Sentiment: Neutral
Rating: 3.0
Distance: 0.5451
--------------------------------------------------
Result 4
Review: Just okay Room is heated up even after using the cooler
Product: Crompton 75 L Desert Air Cooler(White, Teal, ACGC-DAC751)


In [78]:
retrieved_reviews = full_results["documents"][0]

context = "\n\n".join(retrieved_reviews)

print(context[:2000])

Moderate This air cooler does not proper cooling than other metal body cooler,

Moderate This air cooler does not proper cooling than other metal body cooler,

Nice Entire room is not cooling even kept for day and night.avg cooling.review after use of 15 days.normal fan type.

Just okay Room is heated up even after using the cooler

Just okay Room is heated up even after using the cooler

Could be way better Received the product defective, not cooling. Only air coming out

Could be way better Received the product defective, not cooling. Only air coming out

Decent product Not cooling if it is inside Room

Decent product Not cooling if it is inside Room

Expected a better product No cooling. Just it is working as a fan. Even in a small room and for a single person, it is not giving any cool air


In [79]:
prompt = f"""
You are a customer review analysis assistant.

Answer the user's question using only the customer reviews provided below.
Do not make claims that are not supported by the reviews.
If the reviews do not contain enough information, say that there is not enough information.

Customer Reviews:
{context}

Question:
{query}

Answer:
"""

print(prompt[:3000])


You are a customer review analysis assistant.

Answer the user's question using only the customer reviews provided below.
Do not make claims that are not supported by the reviews.
If the reviews do not contain enough information, say that there is not enough information.

Customer Reviews:
Moderate This air cooler does not proper cooling than other metal body cooler,

Moderate This air cooler does not proper cooling than other metal body cooler,

Nice Entire room is not cooling even kept for day and night.avg cooling.review after use of 15 days.normal fan type.

Just okay Room is heated up even after using the cooler

Just okay Room is heated up even after using the cooler

Could be way better Received the product defective, not cooling. Only air coming out

Could be way better Received the product defective, not cooling. Only air coming out

Decent product Not cooling if it is inside Room

Decent product Not cooling if it is inside Room

Expected a better product No cooling. Just it 

In [1]:
import os

print(os.getenv("GEMINI_API_KEY") is not None)

True


In [2]:
from google import genai

client = genai.Client()

print("Gemini client created")

Gemini client created


In [3]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explain sentiment analysis in two simple sentences."
)

print(response.text)

Sentiment analysis is the automated process of determining the emotional tone or opinion expressed in a piece of text, categorizing it as positive, negative, or neutral. It helps businesses and researchers understand public perception, track brand reputation, and analyze customer feedback at scale.


In [4]:
rag_response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(rag_response.text)

NameError: name 'prompt' is not defined

In [5]:
print("context" in globals())

False


In [6]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")

c:\Users\RUSHIKESH\Desktop\customer-review-intelligence\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1893.62it/s]


Embedding model loaded


In [7]:
import chromadb

persistent_client = chromadb.PersistentClient(
    path="../vector_db"
)

full_collection = persistent_client.get_collection(
    name="all_customer_reviews"
)

print("Reviews available:", full_collection.count())

Reviews available: 306316


In [8]:
query = "Air cooler is not cooling the room properly"

query_embedding = embedding_model.encode(query)

full_results = full_collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=10
)

print("Retrieved reviews:", len(full_results["documents"][0]))

Retrieved reviews: 10


In [9]:
retrieved_reviews = full_results["documents"][0]

context = "\n\n".join(retrieved_reviews)

print(context[:1000])

Moderate This air cooler does not proper cooling than other metal body cooler,

Moderate This air cooler does not proper cooling than other metal body cooler,

Nice Entire room is not cooling even kept for day and night.avg cooling.review after use of 15 days.normal fan type.

Just okay Room is heated up even after using the cooler

Just okay Room is heated up even after using the cooler

Could be way better Received the product defective, not cooling. Only air coming out

Could be way better Received the product defective, not cooling. Only air coming out

Decent product Not cooling if it is inside Room

Decent product Not cooling if it is inside Room

Expected a better product No cooling. Just it is working as a fan. Even in a small room and for a single person, it is not giving any cool air


In [10]:
prompt = f"""
You are a customer review analysis assistant.

Answer the user's question using only the customer reviews provided below.

Do not make claims that are not supported by the reviews.
If there is not enough information in the reviews, say that there is not enough information.

Customer Reviews:
{context}

Question:
{query}

Answer:
"""

print(prompt[:1500])


You are a customer review analysis assistant.

Answer the user's question using only the customer reviews provided below.

Do not make claims that are not supported by the reviews.
If there is not enough information in the reviews, say that there is not enough information.

Customer Reviews:
Moderate This air cooler does not proper cooling than other metal body cooler,

Moderate This air cooler does not proper cooling than other metal body cooler,

Nice Entire room is not cooling even kept for day and night.avg cooling.review after use of 15 days.normal fan type.

Just okay Room is heated up even after using the cooler

Just okay Room is heated up even after using the cooler

Could be way better Received the product defective, not cooling. Only air coming out

Could be way better Received the product defective, not cooling. Only air coming out

Decent product Not cooling if it is inside Room

Decent product Not cooling if it is inside Room

Expected a better product No cooling. Just i

In [11]:
from google import genai

client = genai.Client()

In [12]:
rag_response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(rag_response.text)

Yes, according to the customer reviews, the air cooler is not cooling the room properly. Several reviews mention that the entire room is not cooling, the room is heated up even after using the cooler, or that it is not cooling at all and only works like a fan. Some even received defective units that were not cooling.


In [13]:
def ask_reviews(question):
    print("Question:", question)

In [14]:
ask_reviews(
    "What problems are customers facing with air coolers?"
)

Question: What problems are customers facing with air coolers?


In [15]:
def ask_reviews(question):

    print("Question:", question)

    query_embedding = embedding_model.encode(question)

    print("Embedding shape:", query_embedding.shape)

In [16]:
ask_reviews(
    "What problems are customers facing with air coolers?"
)

Question: What problems are customers facing with air coolers?
Embedding shape: (384,)


In [17]:
def ask_reviews(question):

    query_embedding = embedding_model.encode(question)

    results = full_collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=5
    )

    retrieved_reviews = results["documents"][0]

    return retrieved_reviews

In [18]:
reviews = ask_reviews(
    "What problems are customers facing with air coolers?"
)

for review in reviews:
    print(review)
    print("-" * 50)

Does the job Not so good air coolerThat's it
--------------------------------------------------
Does the job Not so good air coolerThat's it
--------------------------------------------------
Hated it! Worst product. Not working after 6 month. Don't buy Crompton Air Coolers.
--------------------------------------------------
Hated it! Worst product. Not working after 6 month. Don't buy Crompton Air Coolers.
--------------------------------------------------
Worst experience ever! Cooling is not good ????????
--------------------------------------------------


In [19]:

def ask_reviews(question):

    query_embedding = embedding_model.encode(question)

    results = full_collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=5
    )

    retrieved_reviews = results["documents"][0]

    context = "\n\n".join(retrieved_reviews)

    return context

In [20]:
context = ask_reviews(
    "What problems are customers facing with air coolers?"
)

print(context)

Does the job Not so good air coolerThat's it

Does the job Not so good air coolerThat's it

Hated it! Worst product. Not working after 6 month. Don't buy Crompton Air Coolers.

Hated it! Worst product. Not working after 6 month. Don't buy Crompton Air Coolers.

Worst experience ever! Cooling is not good ????????


In [21]:
def ask_reviews(question):

    query_embedding = embedding_model.encode(question)

    results = full_collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=5
    )

    retrieved_reviews = results["documents"][0]

    context = "\n\n".join(retrieved_reviews)

    prompt = f"""
You are a customer review analysis assistant.

Answer the user's question using only the customer reviews provided below.

Do not make claims that are not supported by the reviews.
If the reviews do not contain enough information, say that there is not enough information.

Customer Reviews:
{context}

Question:
{question}

Answer:
"""

    return prompt

In [22]:
prompt = ask_reviews(
    "What problems are customers facing with air coolers?"
)

print(prompt[:2000])


You are a customer review analysis assistant.

Answer the user's question using only the customer reviews provided below.

Do not make claims that are not supported by the reviews.
If the reviews do not contain enough information, say that there is not enough information.

Customer Reviews:
Does the job Not so good air coolerThat's it

Does the job Not so good air coolerThat's it

Hated it! Worst product. Not working after 6 month. Don't buy Crompton Air Coolers.

Hated it! Worst product. Not working after 6 month. Don't buy Crompton Air Coolers.

Worst experience ever! Cooling is not good ????????

Question:
What problems are customers facing with air coolers?

Answer:



In [23]:
def ask_reviews(question):

    query_embedding = embedding_model.encode(question)

    results = full_collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=5
    )

    retrieved_reviews = results["documents"][0]

    context = "\n\n".join(retrieved_reviews)

    prompt = f"""
You are a customer review analysis assistant.

Answer the user's question using only the customer reviews provided below.

Do not make claims that are not supported by the reviews.
If the reviews do not contain enough information, say that there is not enough information.

Customer Reviews:
{context}

Question:
{question}

Answer:
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

In [24]:
answer = ask_reviews(
    "What problems are customers facing with air coolers?"
)

print(answer)

Customers are facing the following problems with air coolers:
*   They are "not so good air cooler" in general.
*   Some units are "not working after 6 month".
*   "Cooling is not good".


In [25]:
ask_reviews("What problems are customers facing with air coolers?")

'Customers are facing the following problems with air coolers:\n*   The air cooler is "not so good" or considered the "worst product" by some.\n*   Some units are "not working after 6 months."\n*   The "cooling is not good."'

In [26]:
question = "What problems are customers facing with air coolers?"

query_embedding = embedding_model.encode(question)

negative_results = full_collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5,
    where={"sentiment": "Negative"}
)

for review in negative_results["documents"][0]:
    print(review)
    print("-" * 50)

Hated it! Worst product. Not working after 6 month. Don't buy Crompton Air Coolers.
--------------------------------------------------
Hated it! Worst product. Not working after 6 month. Don't buy Crompton Air Coolers.
--------------------------------------------------
Worst experience ever! Cooling is not good ????????
--------------------------------------------------
Worst experience ever! Worst air cooler
--------------------------------------------------
Terrible product Nathing to much cooling  very very bad company nathing f as St air and to much vinter
--------------------------------------------------


In [27]:
answer = ask_reviews(
    "What are the main product quality complaints from customers?"
)

print(answer)

The main product quality complaints from customers are that the product quality is not good and doesn't reach expectations. Some customers also mentioned unexpected product quality.


In [28]:
answer = ask_reviews(
    "What battery and charging problems are customers reporting?"
)

print(answer)

Customers are reporting "battery issues" and a "battery problem." They are also reporting a "charging problem."


In [29]:
question = "What battery and charging problems are customers reporting?"

query_embedding = embedding_model.encode(question)

test_results = full_collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=10
)

for i, (review, distance) in enumerate(
    zip(
        test_results["documents"][0],
        test_results["distances"][0]
    ),
    start=1
):
    print(f"{i}. {review}")
    print("Distance:", round(distance, 4))
    print("-" * 50)

1. Decent product Battery issues
Distance: 0.5641
--------------------------------------------------
2. Terrible product battery issue
Distance: 0.6749
--------------------------------------------------
3. Horrible Battery problem
Distance: 0.6893
--------------------------------------------------
4. Horrible BATTERY PROBLEM
Distance: 0.6893
--------------------------------------------------
5. Good Charging problem
Distance: 0.6991
--------------------------------------------------
6. Not recommended at all Battery charging issue
Distance: 0.6995
--------------------------------------------------
7. Moderate Battery fault
Distance: 0.7138
--------------------------------------------------
8. Not good Charging problem
Distance: 0.743
--------------------------------------------------
9. Terrible product Charge problem
Distance: 0.7523
--------------------------------------------------
10. Just okay battery problems
Distance: 0.7541
--------------------------------------------------


In [30]:
question = """
Describe specific battery and charging issues customers experience,
such as battery life, charging behavior, battery backup, or battery performance.
"""

query_embedding = embedding_model.encode(question)

better_results = full_collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=10
)

for i, review in enumerate(
    better_results["documents"][0],
    start=1
):
    print(f"{i}. {review}")
    print("-" * 50)

1. Decent product Battery issues
--------------------------------------------------
2. Decent product Battery consumption is fast. Charging is slow.
--------------------------------------------------
3. Bad quality Battery life is 3days only its bad
--------------------------------------------------
4. Moderate Battery fault
--------------------------------------------------
5. Wonderful I have used this laptop for one week and my observations are as below.1. It took 90 to 100 min to charge from 10% to 100%.  ( 5 times tested)2. Battery backup is approx 190 min to 210 min as per this usage => Full brightness, Hotspot in use, Only slideshow application run. (8 times tested)3. After buying this product I have registered it on the HP website and it is found the legal product.4. Keyboard is very very smooth.5. Sound quality is also good. 6. Refresh rate is also good....
--------------------------------------------------
6. Decent product Battery backup very bad
----------------------------

In [31]:
better_context = "\n\n".join(
    better_results["documents"][0]
)

better_prompt = f"""
You are a customer review analysis assistant.

Using only the customer reviews below, summarize the specific battery
and charging problems customers are reporting.

Focus on concrete issues rather than generic statements.
Do not invent information that is not present in the reviews.

Customer Reviews:
{better_context}

Answer:
"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=better_prompt
)

print(response.text)

Customers are reporting the following specific battery and charging problems:

**Battery Problems:**
*   Fast battery consumption.
*   Short battery life or "very bad" battery backup, with one user explicitly stating their battery life is "3 days only."
*   One user observed a specific battery backup of approximately 190-210 minutes (around 3 hours 10 minutes to 3 hours 30 minutes) when used with full brightness, hotspot, and a slideshow application.

**Charging Problems:**
*   Slow charging.
*   One customer reported it takes 90 to 100 minutes to charge from 10% to 100%.
*   Another customer stated that a full charge takes 1 hour.


In [32]:
print(full_collection.count())

306316


In [1]:
from sentence_transformers import SentenceTransformer
import chromadb
from google import genai

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

persistent_client = chromadb.PersistentClient(
    path="../vector_db"
)

full_collection = persistent_client.get_collection(
    name="all_customer_reviews"
)

client = genai.Client()

print("Reviews:", full_collection.count())

c:\Users\RUSHIKESH\Desktop\customer-review-intelligence\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2138.94it/s]


Reviews: 306316


In [2]:
query_expansion = {
    "battery": [
        "battery life",
        "battery backup",
        "battery drain",
        "battery performance",
        "charging",
        "charging speed"
    ],

    "camera": [
        "picture quality",
        "image quality",
        "photo",
        "video",
        "lens"
    ],

    "cooling": [
        "air flow",
        "cooling performance",
        "room cooling",
        "fan speed"
    ],

    "quality": [
        "build quality",
        "material",
        "durability",
        "performance"
    ]
}

print(query_expansion)

{'battery': ['battery life', 'battery backup', 'battery drain', 'battery performance', 'charging', 'charging speed'], 'camera': ['picture quality', 'image quality', 'photo', 'video', 'lens'], 'cooling': ['air flow', 'cooling performance', 'room cooling', 'fan speed'], 'quality': ['build quality', 'material', 'durability', 'performance']}


In [3]:
def expand_query(query):
    expanded_query = query.lower()

    for keyword, related_terms in query_expansion.items():
        if keyword in expanded_query:
            expanded_query += " " + " ".join(related_terms)

    return expanded_query

In [4]:
print(expand_query(
    "What battery problems are customers reporting?"
))

what battery problems are customers reporting? battery life battery backup battery drain battery performance charging charging speed


In [5]:
def ask_reviews(question):

    expanded_query = expand_query(question)

    query_embedding = embedding_model.encode(expanded_query)

    results = full_collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=5
    )

    retrieved_reviews = results["documents"][0]

    context = "\n\n".join(retrieved_reviews)

    prompt = f"""
You are a customer review analysis assistant.

Answer the user's question using only the customer reviews below.

Customer Reviews:
{context}

Question:
{question}

Answer:
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

In [6]:
print(
    ask_reviews(
        "What battery problems are customers reporting?"
    )
)

Customers are reporting that the battery backup is not good, very bad, or non-existent. Specifically, they mention it's much shorter than advertised (e.g., 2 hours instead of 6 hours, or 6 hours instead of 12 hours).


In [7]:
retrieved_reviews = results["documents"][0]

NameError: name 'results' is not defined

In [8]:
def ask_reviews(question):

    expanded_query = expand_query(question)

    query_embedding = embedding_model.encode(expanded_query)

    results = full_collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=10
    )

    retrieved_reviews = results["documents"][0]

    # Remove duplicate reviews
    retrieved_reviews = list(dict.fromkeys(retrieved_reviews))

    context = "\n\n".join(retrieved_reviews)

    prompt = f"""
You are a customer review analysis assistant.

Answer the user's question using only the customer reviews below.

Customer Reviews:
{context}

Question:
{question}

Answer:
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

In [9]:
print(
    ask_reviews(
        "What battery problems are customers reporting?"
    )
)

Customers are reporting that the battery backup is not good, very poor, or very bad. Many state that the actual battery life is significantly less than the advertised hours (e.g., 2 hours instead of 6, or 6 hours instead of 12). One customer also mentioned the battery percentage going up and down continuously.


In [10]:
sample = full_collection.get(
    ids=["review_0"],
    include=["metadatas"]
)

print(sample["metadatas"][0])

{'product': 'Crompton 75 L Desert Air Cooler(White, Teal, ACGC-DAC751)', 'rating': 5.0, 'sentiment': 'Positive'}


In [11]:
where={
    "product": "Crompton 75 L Desert Air Cooler(White, Teal, ACGC-DAC751)"
}

In [12]:
question = "What cooling problems are customers reporting?"

expanded_query = expand_query(question)

query_embedding = embedding_model.encode(expanded_query)

results = full_collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5,
    where={
        "product": "Crompton 75 L Desert Air Cooler(White, Teal, ACGC-DAC751)"
    }
)

for review in results["documents"][0]:
    print(review)
    print("-" * 50)

Bad quality Fan speed is very low in comparison to other brand coolers
--------------------------------------------------
Does the job cooling is not good.speed is well
--------------------------------------------------
Wonderful average product...cooling is not effective.....it's like a simple fan...speed is also slow even max.......
--------------------------------------------------
Great product Cooling is excellent even in large hall. Delivery of the product is really fast I got the item in 2days first time I am experiencing this fast delivery. And packing of the product not good time god there is no damage found in cooler.Con's:-1. Cooling the room is very fast.2. Air flow also good.3. Fan blades are made by metal because of this blades doesn't bend while cooler is running.4. Motor is also made by Crompton.5. Ice adding chamber also really cool.6. Coaster wheels are really ...
--------------------------------------------------
Good fan speed is Not good....expected more......not s